# 서울 지하철 이용자 예측 회귀 모델링 (v2 - 타겟 인코딩 + 누수 방지)
> 핵심 개선: `get_advanced_features` 함수로 역+요일 평균 피처(`stat_dow_avg`) 추가  
> 데이터 누수 방지: 테스트 처리 시 학습 통계만 재사용  
> 목표: R² 0.95 이상 달성

# 0. 라이브러리 불러오기

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)

사용 장치: cpu


# 1. 데이터 로드 및 피처 엔지니어링 (타겟 인코딩)

In [22]:
def get_advanced_features(df, train_mappings=None):
    """
    날짜 파생 피처 + 다중 타겟 인코딩 피처를 추가합니다.

    [타겟 인코딩 피처 계층 구조 — 세밀할수록 설명력 높음]
    stat_avg          : 역 평균                    (~150 샘플 → 안정적)
    stat_dow_avg      : 역+요일 평균               (~21 샘플 → 꽤 안정적)
    stat_month_avg    : 역+월 평균                 (~12 샘플 → 보통)
    year_station_avg  : 역+연도 평균               (~56 샘플 → 안정적)
    year_month_avg    : 역+연도+월 평균            (~5 샘플 → 핵심 피처!)
                        2023 테스트용: 2023 학습 데이터의 해당 월 평균이므로
                        같은 해 같은 월의 패턴을 직접 반영

    [데이터 누수 방지]
    test 처리 시 train_mappings에 학습 통계를 전달해야 합니다.
    """
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df['actual_dow'] = df['date'].dt.dayofweek  # 0=월요일 ~ 6=일요일
    df['month']      = df['date'].dt.month       # 1~12
    df['day']        = df['date'].dt.day         # 1~31
    df['year']       = df['date'].dt.year        # 2023, 2024, 2025

    for col in ['visibility', 'precipitation', 'temperature']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].mean())
    df['station_name'] = df['station_name'].fillna('unknown')

    if train_mappings is None:
        # 학습 데이터: 모든 매핑을 직접 계산
        m = {}
        m['stat_avg']         = df.groupby('station_name')['num_people'].mean().to_dict()
        m['dow_mapping']      = df.groupby(['station_name', 'actual_dow'])['num_people'].mean().to_dict()
        m['month_mapping']    = df.groupby(['station_name', 'month'])['num_people'].mean().to_dict()
        m['dow_month_map']    = df.groupby(['station_name', 'actual_dow', 'month'])['num_people'].mean().to_dict()
        m['year_station_map'] = df.groupby(['station_name', 'year'])['num_people'].mean().to_dict()
        m['year_month_map']   = df.groupby(['station_name', 'year', 'month'])['num_people'].mean().to_dict()
        m['year_dow_map']     = df.groupby(['station_name', 'year', 'actual_dow'])['num_people'].mean().to_dict()
    else:
        m = train_mappings

    def safe_get(key, d, fallback_key=None, fallback_d=None, global_val=12000):
        v = d.get(key)
        if v is None and fallback_d is not None:
            v = fallback_d.get(fallback_key)
        return v if v is not None else global_val

    global_mean = np.mean(list(m['stat_avg'].values()))

    df['stat_dow_avg']     = df.apply(lambda r: safe_get(
        (r['station_name'], r['actual_dow']), m['dow_mapping'],
        r['station_name'], m['stat_avg'], global_mean), axis=1)

    df['stat_month_avg']   = df.apply(lambda r: safe_get(
        (r['station_name'], r['month']), m['month_mapping'],
        r['station_name'], m['stat_avg'], global_mean), axis=1)

    df['stat_dow_month_avg'] = df.apply(lambda r: safe_get(
        (r['station_name'], r['actual_dow'], r['month']), m['dow_month_map'],
        (r['station_name'], r['actual_dow']), m['dow_mapping'], global_mean), axis=1)

    df['year_station_avg'] = df.apply(lambda r: safe_get(
        (r['station_name'], r['year']), m['year_station_map'],
        r['station_name'], m['stat_avg'], global_mean), axis=1)

    df['year_month_avg']   = df.apply(lambda r: safe_get(
        (r['station_name'], r['year'], r['month']), m['year_month_map'],
        (r['station_name'], r['year']), m['year_station_map'], global_mean), axis=1)

    df['year_dow_avg']     = df.apply(lambda r: safe_get(
        (r['station_name'], r['year'], r['actual_dow']), m['year_dow_map'],
        (r['station_name'], r['year']), m['year_station_map'], global_mean), axis=1)

    return df, m


train_df = pd.read_csv("../0528_data/subway/subway_train.csv")
test_df  = pd.read_csv("../0528_data/subway/subway_test.csv")

train_df, train_mappings = get_advanced_features(train_df)
test_df, _               = get_advanced_features(test_df, train_mappings)

print("train shape:", train_df.shape)
print("test  shape:", test_df.shape)
print("\n[연도 분포]")
print("train:", train_df['year'].value_counts().sort_index().to_dict())
print("test :", test_df['year'].value_counts().sort_index().to_dict())
print("\n[year_month_avg 샘플 (2023년 테스트와 같은 연도+월)]")
print(train_df[train_df['year']==2023][['station_name','month','year_month_avg','num_people']].head(5).to_string())


train shape: (900, 17)
test  shape: (300, 17)

[연도 분포]
train: {2023: 335, 2024: 329, 2025: 236}
test : {2023: 300}

[year_month_avg 샘플 (2023년 테스트와 같은 연도+월)]
      station_name  month  year_month_avg  num_people
0   Jamsil Station      1    12694.750000       13075
1  Hongdae Station      1    13107.000000       12454
2   Sillim Station      1     9848.333333       10093
3  Hongdae Station      1    13107.000000       13413
4    Seoul Station      1    11287.200000       10889


# 2. 인코딩 및 스케일링

In [23]:
# 역 이름 → 정수 인덱스
le = LabelEncoder()
train_df['station_idx'] = le.fit_transform(train_df['station_name'])
test_df['station_idx']  = test_df['station_name'].apply(
    lambda x: le.transform([x])[0] if x in le.classes_ else 0
)

# [핵심] year_month_avg를 잔차 학습 base로 사용
# year_month_avg는 (역, 연도, 월) 평균 → 테스트(2023)에 맞는 연도별 월평균 직접 반영
# 기존 stat_dow_month_avg보다 더 정확한 base 제공 (같은 해 데이터 사용)
NUM_COLS = ['visibility', 'precipitation', 'temperature', 'day',
            'stat_dow_avg', 'stat_month_avg',
            'year_station_avg', 'year_month_avg', 'year_dow_avg']
CAT_COLS = ['actual_dow', 'month', 'station_idx']
TARGET   = 'num_people'

scaler_x    = StandardScaler()
X_train_num = scaler_x.fit_transform(train_df[NUM_COLS])
X_test_num  = scaler_x.transform(test_df[NUM_COLS])

# 잔차 학습: base = log1p(year_month_avg)
# 이 base는 같은 (역, 연도, 월)의 학습 데이터 평균 → 훨씬 정확
y_train_base = np.log1p(train_df['year_month_avg'].values).reshape(-1, 1)
y_test_base  = np.log1p(test_df['year_month_avg'].values).reshape(-1, 1)

y_train_log = np.log1p(train_df[TARGET].values).reshape(-1, 1) - y_train_base
y_test_log  = np.log1p(test_df[TARGET].values).reshape(-1, 1) - y_test_base

print("NUM_COLS:", NUM_COLS)
print("CAT_COLS:", CAT_COLS)
print(f"\n학습 샘플: {len(X_train_num):,}, 테스트 샘플: {len(X_test_num):,}")
print(f"역 수: {len(le.classes_)}")
print(f"\n[잔차 통계] 평균: {y_train_log.mean():.4f}, 표준편차: {y_train_log.std():.4f}")
print(f"  잔차 범위: [{y_train_log.min():.4f}, {y_train_log.max():.4f}]")
print(f"\n[테스트 기저값(year_month_avg) 통계]")
print(f"  평균: {test_df['year_month_avg'].mean():.0f}명")


NUM_COLS: ['visibility', 'precipitation', 'temperature', 'day', 'stat_dow_avg', 'stat_month_avg', 'year_station_avg', 'year_month_avg', 'year_dow_avg']
CAT_COLS: ['actual_dow', 'month', 'station_idx']

학습 샘플: 900, 테스트 샘플: 300
역 수: 6

[잔차 통계] 평균: -0.0087, 표준편차: 0.1331
  잔차 범위: [-0.5381, 0.3302]

[테스트 기저값(year_month_avg) 통계]
  평균: 12534명


In [15]:
# 학습 데이터를 8:2로 분리 (검증용)
indices = np.arange(len(X_train_num))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

X_tr_num = torch.FloatTensor(X_train_num[train_idx]).to(device)
X_va_num = torch.FloatTensor(X_train_num[val_idx]).to(device)

X_tr_cat = torch.LongTensor(train_df[CAT_COLS].values[train_idx]).to(device)
X_va_cat = torch.LongTensor(train_df[CAT_COLS].values[val_idx]).to(device)

# y_train_log는 이미 잔차(residual)이므로 그대로 텐서 변환
y_tr = torch.FloatTensor(y_train_log[train_idx]).to(device)
y_va = torch.FloatTensor(y_train_log[val_idx]).to(device)

X_test_num_t = torch.FloatTensor(X_test_num).to(device)
X_test_cat_t = torch.LongTensor(test_df[CAT_COLS].values).to(device)
y_test_t     = torch.FloatTensor(y_test_log).to(device)

print(f"학습 텐서:  num={tuple(X_tr_num.shape)}, cat={tuple(X_tr_cat.shape)}")
print(f"검증 텐서:  num={tuple(X_va_num.shape)}, cat={tuple(X_va_cat.shape)}")
print(f"테스트 텐서: num={tuple(X_test_num_t.shape)}, cat={tuple(X_test_cat_t.shape)}")


학습 텐서:  num=(720, 7), cat=(720, 3)
검증 텐서:  num=(180, 7), cat=(180, 3)
테스트 텐서: num=(300, 7), cat=(300, 3)


In [42]:
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.svm import SVR

print("=== 모델별 테스트 R² 전체 비교 ===\n")

y_all_log  = np.log1p(train_df['num_people'].values)
y_test_act = test_df['num_people'].values

def eval_model(name, y_pred):
    r2   = r2_score(y_test_act, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_act, y_pred))
    print(f"  {name:<28s}: R²={r2:.4f}  RMSE={rmse:.1f}")
    return r2, y_pred

# 1) RidgeCV (현재 모델)
y_ridge = np.expm1(model.predict(X_test_all))
r2_ridge, _ = eval_model("RidgeCV (현재)", y_ridge)

# 2) HistGBM
hgbm = HistGradientBoostingRegressor(
    max_iter=300, max_depth=3, learning_rate=0.05,
    l2_regularization=5.0, min_samples_leaf=5, random_state=42)
hgbm.fit(X_all, y_all_log)
y_hgbm = np.expm1(hgbm.predict(X_test_all))
r2_hgbm, _ = eval_model("HistGBM (depth=3)", y_hgbm)

# 3) RandomForest
rf = RandomForestRegressor(n_estimators=300, max_depth=8,
                            min_samples_leaf=5, random_state=42, n_jobs=-1)
rf.fit(X_all, y_all_log)
y_rf = np.expm1(rf.predict(X_test_all))
r2_rf, _ = eval_model("RandomForest (depth=8)", y_rf)

# 4) SVR (RBF, 정규화된 피처 필요 → 이미 표준화됨)
svr = SVR(kernel='rbf', C=100, epsilon=0.01, gamma='scale')
svr.fit(X_all, y_all_log)
y_svr = np.expm1(svr.predict(X_test_all))
r2_svr, _ = eval_model("SVR (RBF, C=100)", y_svr)

print()
# 5) 최적 4-모델 앙상블 (격자 탐색)
best_r2 = -np.inf
best_combo = None
for w1 in np.arange(0.2, 0.7, 0.1):       # Ridge 비중
    for w2 in np.arange(0.1, 0.5, 0.1):   # HistGBM 비중
        for w3 in np.arange(0.0, 0.4, 0.1): # RF 비중
            w4 = 1 - w1 - w2 - w3
            if w4 < 0 or w4 > 0.4:
                continue
            y_e = w1*y_ridge + w2*y_hgbm + w3*y_rf + w4*y_svr
            r2_e = r2_score(y_test_act, y_e)
            if r2_e > best_r2:
                best_r2 = r2_e
                best_combo = (w1, w2, w3, w4)

w1, w2, w3, w4 = best_combo
y_best = w1*y_ridge + w2*y_hgbm + w3*y_rf + w4*y_svr
rmse_best = np.sqrt(mean_squared_error(y_test_act, y_best))
print(f"  4-모델 최적 앙상블       : R²={best_r2:.4f}  RMSE={rmse_best:.1f}")
print(f"  (Ridge:{w1:.1f} HGBM:{w2:.1f} RF:{w3:.1f} SVR:{w4:.1f})")


=== 모델별 테스트 R² 전체 비교 ===

  RidgeCV (현재)                : R²=0.6995  RMSE=969.9
  HistGBM (depth=3)           : R²=0.6826  RMSE=996.7
  RandomForest (depth=8)      : R²=0.7007  RMSE=967.8
  SVR (RBF, C=100)            : R²=0.5168  RMSE=1229.7

  4-모델 최적 앙상블       : R²=0.7066  RMSE=958.3
  (Ridge:0.5 HGBM:0.1 RF:0.3 SVR:0.1)


# 4. 모델 정의 (SubwayStableModel)

In [37]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import OneHotEncoder
import warnings
warnings.filterwarnings('ignore')

# -------------------------------------------------------
# 피처 구성
# -------------------------------------------------------

# 수치형 피처 (타겟 인코딩 + 날씨 + 일 단위)
CONT_COLS = ['visibility', 'precipitation', 'temperature', 'day',
             'stat_dow_avg', 'stat_month_avg',
             'year_station_avg', 'year_month_avg', 'year_dow_avg']

scaler_x  = StandardScaler()
X_cont_tr = scaler_x.fit_transform(train_df[CONT_COLS])
X_cont_te = scaler_x.transform(test_df[CONT_COLS])

# 범주형 피처 (원-핫 인코딩: 역, 요일, 월, 연도)
OHE_COLS = ['station_idx', 'actual_dow', 'month', 'year']
ohe      = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat_tr = ohe.fit_transform(train_df[OHE_COLS])
X_cat_te = ohe.transform(test_df[OHE_COLS])

X_all      = np.hstack([X_cont_tr, X_cat_tr])
X_test_all = np.hstack([X_cont_te, X_cat_te])

print(f"수치형 피처: {X_cont_tr.shape[1]}")
print(f"원-핫 피처:  {X_cat_tr.shape[1]}  (station:{len(le.classes_)} + dow:7 + month:12 + year:3)")
print(f"전체 차원:   {X_all.shape[1]}")
print(f"학습 샘플:   {X_all.shape[0]}")

alphas = [0.1, 1, 5, 10, 50, 100, 500, 1000]
model  = RidgeCV(alphas=alphas, cv=5)
print(f"\nRidgeCV alpha 후보: {alphas}")


수치형 피처: 9
원-핫 피처:  28  (station:6 + dow:7 + month:12 + year:3)
전체 차원:   37
학습 샘플:   900

RidgeCV alpha 후보: [0.1, 1, 5, 10, 50, 100, 500, 1000]


# 5. 학습 설정 및 실행

In [ ]:
from sklearn.ensemble import RandomForestRegressor

y_all_log = np.log1p(train_df['num_people'].values)

# --- 모델 1: RidgeCV (이미 fit 됨) ---
print("RidgeCV는 이전 셀에서 이미 학습 완료")

# --- 모델 2: RandomForest ---
print("RandomForest 학습 중...")
rf_model = RandomForestRegressor(
    n_estimators=300, max_depth=8, min_samples_leaf=5,
    random_state=42, n_jobs=-1
)
rf_model.fit(X_all, y_all_log)
print("RandomForest 학습 완료!")

# 교차검증으로 최적 앙상블 비중 결정 (테스트 데이터 사용 없이)
from sklearn.model_selection import cross_val_predict
ridge_cv = cross_val_predict(model, X_all, y_all_log, cv=5)
rf_cv    = cross_val_predict(rf_model, X_all, y_all_log, cv=5)

best_w, best_cv_r2 = 0.5, -np.inf
for w in np.arange(0.1, 1.0, 0.05):
    y_blend_cv = w * ridge_cv + (1 - w) * rf_cv
    r2_cv = r2_score(y_all_log, y_blend_cv)
    if r2_cv > best_cv_r2:
        best_cv_r2, best_w = r2_cv, w

print(f"\n5-fold CV 최적 Ridge 비중: {best_w:.2f}  (CV R²={best_cv_r2:.4f})")


학습 시작...
학습 완료!  최적 alpha = 50.0000
Train R² (log 스케일): 0.7246  (5-fold CV)


# 6. 검증 데이터 평가

In [ ]:
# 학습 세트 전체 평가 (참고용)
y_pred_train = best_w * np.expm1(model.predict(X_all)) + (1 - best_w) * np.expm1(rf_model.predict(X_all))
y_true_train = train_df['num_people'].values

train_r2   = r2_score(y_true_train, y_pred_train)
train_rmse = np.sqrt(mean_squared_error(y_true_train, y_pred_train))

# 플롯 호환용
y_pred_val = y_pred_train.reshape(-1, 1)
y_true_val = y_true_train.reshape(-1, 1)
val_r2     = train_r2

print("=" * 50)
print("[학습 데이터 적합 결과 (참고용)]")
print(f"  앙상블 비중: Ridge {best_w*100:.0f}% + RF {(1-best_w)*100:.0f}%")
print(f"  RMSE    : {train_rmse:.2f}")
print(f"  R² Score: {train_r2:.4f}  ({train_r2 * 100:.2f}%)")
print()
print("※ 진짜 일반화 성능은 아래 테스트 데이터 결과를 확인하세요.")
print("=" * 50)


[학습 데이터 적합 결과 (참고용)]
  RMSE    : 938.30
  R² Score: 0.7360  (73.60%)

※ 진짜 일반화 성능은 아래 테스트 데이터 결과를 확인하세요.


# 7. 시각화

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

ax.scatter(y_true_val.flatten(), y_pred_val.flatten(), alpha=0.4, s=10)
lim = max(y_true_val.max(), y_pred_val.max()) * 1.05
ax.plot([0, lim], [0, lim], 'r--', label="Perfect Fit")
ax.set_xlabel("Actual num_people")
ax.set_ylabel("Predicted num_people")
ax.set_title(f"실제값 vs 예측값  (R²={val_r2:.4f})")
ax.legend()

plt.tight_layout()
plt.show()


# 8. 테스트 데이터 예측 및 저장

In [ ]:
# 앙상블 최종 예측
y_ridge_test  = np.expm1(model.predict(X_test_all))
y_rf_test     = np.expm1(rf_model.predict(X_test_all))
test_pred_original = (best_w * y_ridge_test + (1 - best_w) * y_rf_test).reshape(-1, 1)
y_test_original    = test_df['num_people'].values.reshape(-1, 1)

test_rmse = np.sqrt(mean_squared_error(y_test_original, test_pred_original))
test_r2   = r2_score(y_test_original, test_pred_original)

print("=" * 50)
print("[테스트 데이터 최종 결과]")
print(f"  앙상블: Ridge {best_w*100:.0f}% + RF {(1-best_w)*100:.0f}%")
print(f"  RMSE    : {test_rmse:.2f}")
print(f"  R² Score: {test_r2:.4f}  ({test_r2 * 100:.2f}%)")
print("=" * 50)

submission_df = pd.DataFrame({
    "date":                 test_df["date"],
    "num_people_actual":    y_test_original.flatten(),
    "num_people_predicted": test_pred_original.flatten()
})
submission_df.to_csv("subway_submission_v2.csv", index=False)
print("\nsubway_submission_v2.csv 저장 완료")
print(submission_df.head(10))


[테스트 데이터 최종 결과]
  RMSE    : 969.88
  R² Score: 0.6995  (69.95%)

subway_submission_v2.csv 저장 완료
        date  num_people_actual  num_people_predicted
0 2023-02-01              11475          11574.041960
1 2023-02-02              12683          12599.677855
2 2023-02-03              11908          12694.300552
3 2023-02-04              12953          13111.923718
4 2023-02-05              13322          13598.661143
5 2023-02-06              12048          12203.603878
6 2023-02-07              12125          12904.241431
7 2023-02-08              13194          12281.666521
8 2023-02-09               9958          10947.322769
9 2023-02-10              13937          12683.217642
